Bài toán dễ hiểu

Một cửa hàng có dữ liệu:
Sản phẩm	Số lượng	Đơn giá
Bút	        2	        5.000
Bút	        1	        5.000
Vở	        3	        10.000

Cửa hàng muốn:
1. Tính doanh thu từng sản phẩm.
2. Áp dụng quy tắc riêng: giá trị một dòng hàng từ 30.000 đồng trở lên được giảm 10%; thấp hơn thì giữ nguyên.

Quy tắc giảm giá này là ví dụ do AI tạo ra, không phải nội dung hoặc demo của tác giả.

In [ ]:
# Nạp mô-đun sys thuộc thư viện chuẩn của Python.
# Mô-đun này cung cấp thông tin về trình thông dịch Python đang chạy.
import sys

# Nạp hàm version từ mô-đun importlib.metadata thuộc thư viện chuẩn.
# Hàm này đọc phiên bản của một gói đã được cài trong môi trường Python hiện tại.
from importlib.metadata import version

# Nạp thư viện PyArrow và đặt tên ngắn là pa.
# PyArrow là thư viện cung cấp khả năng làm việc với Apache Arrow trong Python.
# Ví dụ: pa.table(...) tạo bảng, pa.array(...) tạo mảng dữ liệu.
import pyarrow as pa

# Nạp mô-đun tính toán của PyArrow và đặt tên ngắn là pc.
# Mô-đun này có các hàm như nhân, so sánh, chuyển kiểu và chọn giá trị theo điều kiện.
# Ví dụ: pc.multiply(...) nhân các giá trị trong mảng Arrow.
import pyarrow.compute as pc

# Nạp ba thành phần cần dùng từ thư viện DataFusion:
#
# SessionConfig:
# Tạo cấu hình cho môi trường truy vấn.
# Ví dụ: thiết lập số phân vùng đích bằng with_target_partitions(...).
#
# SessionContext:
# Tạo môi trường để đăng ký bảng, đăng ký hàm và gửi câu truy vấn.
# Đối tượng này là điểm tương tác chính giữa chương trình Python và DataFusion.
#
# udf:
# Tạo đối tượng hàm do người dùng định nghĩa.
# Đối tượng này chứa hàm Python cùng thông tin về kiểu đầu vào,
# kiểu đầu ra, đặc tính và tên hàm sử dụng trong SQL.
from datafusion import SessionConfig, SessionContext, udf


print('Python:', sys.version)
print('DataFusion:', version('datafusion'))
print('PyArrow:', version('pyarrow'))

Python: 3.13.15 (main, Aug  5 2026, 12:25:43) [Clang 21.0.0 (clang-2100.1.1.101)]
DataFusion: 54.0.0
PyArrow: 25.0.1


Apache Arrow ra đời từ nhu cầu về một bộ tiêu chuẩn xoay quanh việc biểu diễn dữ liệu dạng bảng và trao đổi dữ liệu giữa các hệ thống. Việc áp dụng các tiêu chuẩn này giúp giảm chi phí tính toán cho việc tuần tự hóa/giải tuần tự hóa dữ liệu và chi phí triển khai trên các hệ thống được lập trình bằng các ngôn ngữ khác nhau

pa.table({...}) là lệnh tạo một bảng dữ liệu theo cách biểu diễn của Apache Arrow trong bộ nhớ.

In [61]:
# Tạo bảng dữ liệu bán hàng mẫu bằng Apache Arrow.
# Mỗi khóa trong từ điển là tên một cột.
# Các phần tử có cùng vị trí trong các cột tạo thành một dòng dữ liệu.
# Xem hàm table ở https://arrow.apache.org/docs/python/generated/pyarrow.Table.html
sales_data = pa.table({
    "line_id": pa.array([1, 2, 3], type=pa.int64()),
    "product_name": pa.array(["Bút", "Bút", "Vở"], type=pa.string()),
    "quantity": pa.array([2, 1, 3], type=pa.int64()),
    "unit_price_vnd": pa.array([5000, 5000, 10000], type=pa.int64()),
})

# Hiển thị cấu trúc cột, kiểu dữ liệu và các giá trị trong bảng.
print(sales_data)

pyarrow.Table
line_id: int64
product_name: string
quantity: int64
unit_price_vnd: int64
----
line_id: [[1,2,3]]
product_name: [["Bút","Bút","Vở"]]
quantity: [[2,1,3]]
unit_price_vnd: [[5000,5000,10000]]


In [62]:
# Tạo đối tượng cấu hình cho phiên truy vấn DataFusion.
# SessionConfig chứa các thiết lập ảnh hưởng đến cách DataFusion lập kế hoạch
# và thực thi truy vấn.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html
config = SessionConfig()

# Đặt số phân vùng đích bằng 1.
# Phân vùng là phần dữ liệu hoặc công việc mà DataFusion có thể xử lý riêng.
# Đặt 1 phân vùng giúp ví dụ nhỏ có kế hoạch đơn giản, dễ quan sát hơn.
# Giá trị này không phải số dòng và cũng không phải số nhóm trong GROUP BY.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionConfig.with_target_partitions
config = config.with_target_partitions(1)

# Tạo môi trường truy vấn DataFusion bằng cấu hình đã chuẩn bị.
# ctx sẽ quản lý các bảng, hàm và câu truy vấn được sử dụng trong phiên làm việc.
# Nếu không có ctx, chương trình chưa có môi trường để đăng ký dữ liệu
# và gửi câu SQL cho DataFusion.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext
ctx = SessionContext(config)

# Chuyển bảng Arrow sales_data thành các RecordBatch.
# RecordBatch là một lô dữ liệu gồm nhiều cột Arrow có cùng số dòng.
# DataFusion xử lý dữ liệu theo các lô thay vì bắt buộc xử lý từng dòng riêng lẻ.
# Phương thức to_batches() trả về một danh sách các lô dữ liệu.
# Tài liệu:
# https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_batches
sales_batches = sales_data.to_batches()

# Đăng ký các RecordBatch trong bộ nhớ với DataFusion dưới tên bảng "sales".
# Sau khi đăng ký, câu SQL có thể sử dụng:
#
#     select * from sales
#
# Danh sách bên ngoài biểu diễn các phân vùng dữ liệu.
# Ở đây chỉ có một phân vùng, bên trong chứa các lô do sales_data.to_batches() tạo ra.
# Dòng này cần thiết vì DataFusion chưa biết tên "sales" nếu bảng Arrow
# chỉ tồn tại trong biến sales_data của Python.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.register_record_batches
ctx.register_record_batches("sales", [sales_batches])

# Tạo truy vấn SQL trên bảng sales đã đăng ký.
# SELECT * lấy tất cả các cột.
# FROM sales cho biết nguồn dữ liệu là bảng sales.
# ORDER BY line_id sắp xếp theo mã dòng tăng dần,
# giúp kết quả hiển thị ổn định và dễ đối chiếu với dữ liệu ban đầu.
# Lời gọi sql() tạo đối tượng truy vấn DataFusion; câu truy vấn
# được thực thi khi gọi phương thức hiển thị hoặc lấy kết quả.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.sql
query = ctx.sql("""
    select * 
    from sales
""")

# Thực thi truy vấn và hiển thị kết quả dưới dạng bảng.
# show() giúp quan sát trực tiếp kết quả trong notebook.
# Nếu chỉ tạo query mà chưa gọi show() hoặc collect(),
# dữ liệu chưa được lấy ra để hiển thị.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.show
query.show()

DataFrame()
+---------+--------------+----------+----------------+
| line_id | product_name | quantity | unit_price_vnd |
+---------+--------------+----------+----------------+
| 1       | Bút          | 2        | 5000           |
| 2       | Bút          | 1        | 5000           |
| 3       | Vở           | 3        | 10000          |
+---------+--------------+----------+----------------+


In [63]:
# Tạo truy vấn đọc bảng sales đã đăng ký với DataFusion.
# Ba dấu nháy đơn cho phép viết chuỗi SQL trên nhiều dòng để dễ đọc.
#
# Các cột đầu tiên được giữ lại để đối chiếu với dữ liệu ban đầu.
# quantity * unit_price_vnd tính giá trị của từng dòng bán hàng.
# AS line_total_vnd đặt tên cho cột kết quả của phép nhân.
# Cột này chỉ nằm trong kết quả truy vấn, không được thêm vào sales_data.
#
# ORDER BY line_id sắp xếp kết quả theo mã dòng tăng dần.
# Cần chỉ định thứ tự vì SQL không bảo đảm thứ tự kết quả nếu thiếu ORDER BY.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.sql
query = ctx.sql('''
    select
        line_id,
        product_name,
        quantity,
        unit_price_vnd,
        quantity * unit_price_vnd as line_total_vnd
    from sales
    order by line_id
''')

# Thực thi truy vấn và in kết quả dưới dạng bảng.
# Dùng bước này để kiểm tra giá trị từng dòng trước khi gom nhóm và tính tổng.
# show() mặc định hiển thị tối đa 20 dòng; dữ liệu mẫu có 3 dòng nên hiện đủ.
# Phương thức này phục vụ quan sát, không lưu kết quả vào một biến hay tệp.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.show
query.show()

DataFrame()
+---------+--------------+----------+----------------+----------------+
| line_id | product_name | quantity | unit_price_vnd | line_total_vnd |
+---------+--------------+----------+----------------+----------------+
| 1       | Bút          | 2        | 5000           | 10000          |
| 2       | Bút          | 1        | 5000           | 5000           |
| 3       | Vở           | 3        | 10000          | 30000          |
+---------+--------------+----------+----------------+----------------+


In [ ]:
# Lưu yêu cầu truy vấn dưới dạng chuỗi SQL.
# quantity * unit_price_vnd tính giá trị từng dòng bán hàng.
# sum(...) cộng các giá trị trong cùng một nhóm sản phẩm.
# group by product_name gom những dòng có cùng tên sản phẩm.
# order by product_name sắp xếp kết quả theo tên sản phẩm tăng dần.

revenue_sql = '''
    select
        product_name,
        sum(quantity * unit_price_vnd) as revenue
    from sales
    group by product_name
    order by product_name
'''

# Gửi chuỗi SQL cho DataFusion để tạo đối tượng truy vấn.
# DataFusion phân tích câu SQL và xác định các phép xử lý cần thực hiện.
# Biến revenue_query không chứa kết quả doanh thu;
# nó đại diện cho truy vấn sẽ được thực thi khi lấy kết quả.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.sql
revenue_query = ctx.sql(revenue_sql)

# Hiển thị kế hoạch truy vấn để quan sát DataFusion sẽ xử lý yêu cầu ra sao.
# Với phiên bản đang sử dụng, explain() tự hiển thị kế hoạch;
# không cần gọi thêm show() sau explain().
#
# logical_plan: mô tả các phép xử lý cần thực hiện.
# physical_plan: mô tả các thành phần thực thi được chọn để làm những việc đó.
#
# explain() không chạy phép tổng hợp để trả về doanh thu.
# Nó cũng không cung cấp thời gian chạy thực tế của truy vấn doanh thu.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.explain
revenue_query.show()
revenue_query.explain()

# KẾ HOẠCH LÔ-GIC: logical_plan
#
# Đọc cây kế hoạch từ các dòng thụt sâu nhất lên phía trên:
# nguồn dữ liệu ở dưới cung cấp đầu vào cho phép xử lý phía trên.
#
# 1. TableScan: sales projection=[product_name, quantity, unit_price_vnd]
#    Đọc bảng sales và chỉ lấy ba cột cần cho truy vấn.
#    Cột line_id không cần thiết nên không xuất hiện trong danh sách này.
#    Điều này cho thấy kế hoạch chỉ yêu cầu những cột có liên quan.
#    Tuy nhiên, chưa thể kết luận lượng đọc đĩa giảm vì dữ liệu đang ở bộ nhớ.
#
# 2. Aggregate: groupBy=[[sales.product_name]], aggr=[[sum(...)]]
#    Gom các dòng theo product_name.
#    Trong mỗi nhóm, tính tổng quantity * unit_price_vnd.
#    Hai dòng Bút được cộng chung: 2 * 5000 + 1 * 5000 = 15000.
#    Nhóm Vở có một dòng: 3 * 10000 = 30000.
#    Phép nhân nằm trong biểu thức tổng hợp; kế hoạch này không có
#    một bước tính phép nhân riêng biệt phía dưới Aggregate.
#
# 3. Projection: ... AS revenue
#    Chọn các cột đầu ra và đặt tên revenue cho kết quả tổng hợp.
#    Ở đây không phải tính lại tổng doanh thu lần thứ hai.
#
# 4. Sort: sales.product_name ASC NULLS LAST
#    Sắp xếp kết quả theo tên sản phẩm tăng dần.
#    ASC nghĩa là tăng dần.
#    NULLS LAST nghĩa là giá trị thiếu được đặt cuối nếu có.


# KẾ HOẠCH VẬT LÝ: physical_plan
#
# Kế hoạch này cho biết các thành phần thực thi mà DataFusion đã chọn.
# Đọc từ dưới lên giống kế hoạch lô-gic.
#
# 1. DataSourceExec: partitions=1, partition_sizes=[1]
#    Cung cấp dữ liệu từ bảng Arrow đã đăng ký.
#    Có một phân vùng đầu vào.
#    Với nguồn dữ liệu trong bộ nhớ của ví dụ này, [1] biểu thị phân vùng đó chứa một lô RecordBatch, không phải một dòng dữ liệu.
#
# 2. AggregateExec: mode=Single
#    Thực hiện gom nhóm và tính tổng.
#    Single nghĩa là tổng hợp trong một giai đoạn,
#    thay vì tách thành giai đoạn tính tổng cục bộ và hợp nhất kết quả.
#    Không có nghĩa là kết quả chỉ có một nhóm hoặc một dòng.
#
# 3. ProjectionExec
#    Tạo các cột đầu ra theo danh sách SELECT và tên đã đặt.
#    product_name@0 là cột ở vị trí 0 trong đầu vào của thành phần này.
#    Biểu thức tổng hợp có @1 nằm ở vị trí 1.
#    Các vị trí này tính từ 0 và không phải mã dòng.
#
# 4. SortExec
#    Thực hiện sắp xếp theo product_name để đáp ứng ORDER BY.
#    preserve_partitioning=[false] cho biết phép sắp xếp này
#    không được cấu hình để giữ riêng từng phân vùng đầu vào.
#    Trong ví dụ chỉ có một phân vùng, không cần phân tích sâu thuộc tính này.


# Ý NGHĨA CỦA VIỆC HIỂN THỊ KẾ HOẠCH
#
# Kết quả cho thấy DataFusion đã biến câu SQL thành các phép xử lý
# và chọn các thành phần thực thi tương ứng.
# Chương trình không phải tự viết thuật toán gom nhóm hoặc sắp xếp.
# Đây là bằng chứng minh họa việc sử dụng lại bộ máy truy vấn DataFusion.
#
# Kế hoạch chưa chứng minh phép tính cho kết quả đúng.
# Muốn kiểm tra kết quả, cần chạy revenue_query.show() hoặc collect().
#
# Kế hoạch không chứng minh DataFusion nhanh hơn DuckDB.
# Muốn đánh giá tốc độ, cần một thí nghiệm đo thời gian riêng.
#
# Ví dụ này chưa thể hiện khả năng mở rộng bằng hàm riêng.
# Phần đó được minh họa khi đăng ký và gọi hàm do người dùng định nghĩa.

DataFrame()
+--------------+---------+
| product_name | revenue |
+--------------+---------+
| Bút          | 15000   |
| Vở           | 30000   |
+--------------+---------+
DataFrame()
+---------------+-------------------------------------------------------------------------------------------------------------------------+
| plan_type     | plan                                                                                                                    |
+---------------+-------------------------------------------------------------------------------------------------------------------------+
| logical_plan  | Sort: sales.product_name ASC NULLS LAST                                                                                 |
|               |   Projection: sales.product_name, sum(sales.quantity * sales.unit_price_vnd) AS revenue                                 |
|               |     Aggregate: groupBy=[[sales.product_name]], aggr=[[sum(sales.quantity * sales.unit_price_vnd)

In [48]:
# Thực thi truy vấn doanh thu và lấy toàn bộ kết quả về Python.
# collect() trả về danh sách các lô dữ liệu Arrow (RecordBatch).
# Đây là bước thực sự yêu cầu DataFusion xử lý dữ liệu,
# khác với explain() chỉ hiển thị kế hoạch xử lý.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.collect
revenue_batches = revenue_query.collect()

# Tạo một bảng Arrow từ các lô kết quả vừa nhận.
# Bước này giúp thao tác với toàn bộ kết quả dưới một đối tượng bảng.
# Nó không tính lại doanh thu; doanh thu đã được DataFusion tính ở bước collect().
#
# Tài liệu:
# https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.from_batches
revenue_result = pa.Table.from_batches(revenue_batches)

# Hiển thị bảng kết quả, gồm tên cột, kiểu dữ liệu và các giá trị.
# print() chỉ hiển thị bảng đã lấy về, không chạy lại truy vấn.
print(revenue_result)

# Chuyển bảng Arrow thành từ điển Python để dễ kiểm tra.
# Mỗi khóa là tên cột; mỗi giá trị là danh sách dữ liệu của cột đó.
#
# Tài liệu:
# https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pydict
actual_revenue = revenue_result.to_pydict()

# Khai báo kết quả đúng được tính độc lập bằng tay từ dữ liệu mẫu.
# Bút: 2 * 5000 + 1 * 5000 = 15000.
# Vở: 3 * 10000 = 30000.
# Tên các khóa phải trùng với tên cột trong kết quả SQL.
expected_revenue = {
    "product_name": ["Bút", "Vở"],
    "revenue": [15000, 30000],
}

# Kiểm tra kết quả thực tế có bằng kết quả mong đợi không.
# Nếu bằng nhau: chương trình tiếp tục, assert không in thông báo.
# Nếu khác nhau: phát sinh lỗi AssertionError để báo kiểm tra thất bại.
# Thứ tự các phần tử cũng được kiểm tra, nên câu SQL có ORDER BY product_name.
assert actual_revenue == expected_revenue, (
    f"Kết quả không đúng. "
    f"Mong đợi: {expected_revenue}; thực tế: {actual_revenue}"
)

pyarrow.Table
product_name: string
revenue: int64
----
product_name: [["Bút","Vở"]]
revenue: [[15000,30000]]


In [50]:
# Định nghĩa hàm nhận một mảng Arrow chứa giá trị các dòng bán hàng.
# line_totals_vnd là mảng nhiều giá trị, không phải một số tiền đơn lẻ.
# Hàm trả về một mảng tiền sau giảm, giữ nguyên số phần tử và thứ tự.
def calculate_discounted_totals(line_totals_vnd):

    # So sánh từng phần tử với ngưỡng 30000.
    # Kết quả là một mảng điều kiện đúng hoặc sai.
    # Ví dụ: [10000, 5000, 30000] → [False, False, True].
    #
    # Tài liệu:
    # https://arrow.apache.org/docs/python/generated/pyarrow.compute.greater_equal.html
    discount_eligible = pc.greater_equal(line_totals_vnd, 30000)

    # Tính số tiền còn lại sau khi giảm 10%.
    # Nhân với 0.9 nghĩa là giữ lại 90% giá trị ban đầu.
    # Phép nhân được áp dụng cho từng phần tử của mảng.
    # Ví dụ: [10000, 5000, 30000] → [9000.0, 4500.0, 27000.0].
    #
    # Tài liệu:
    # https://arrow.apache.org/docs/python/generated/pyarrow.compute.multiply.html
    discounted_totals = pc.multiply(line_totals_vnd, 0.9)

    # Chuyển các giá trị ban đầu sang kiểu số thực 64 bit.
    # Bước này làm rõ rằng cả nhánh giảm giá và nhánh giữ nguyên
    # đều cung cấp dữ liệu cùng kiểu số thực.
    # Chuyển kiểu không áp dụng giảm giá.
    #
    # Tài liệu:
    # https://arrow.apache.org/docs/python/generated/pyarrow.compute.cast.html
    original_totals = pc.cast(line_totals_vnd, pa.float64())

    # Chọn kết quả theo điều kiện tại từng vị trí:
    # - Điều kiện True: lấy giá trị từ discounted_totals.
    # - Điều kiện False: lấy giá trị từ original_totals.
    # Các mảng trên đã được tính trước; if_else chọn giá trị tương ứng,
    # không phải chỉ tính phép nhân ở những dòng đủ điều kiện.
    #
    # pc.if_else(điều_kiện, giá_trị_khi_đúng, giá_trị_khi_sai)
    # 
    # Tài liệu:
    # https://arrow.apache.org/docs/python/generated/pyarrow.compute.if_else.html
    return pc.if_else(
        discount_eligible, # Mảng điều kiện: [False, False, True].
        discounted_totals, # Các giá trị để chọn khi điều kiện là True.
        original_totals, # Các giá trị để chọn khi điều kiện là False.
    )


# Tạo mảng giá trị dòng hàng để thử hàm độc lập.
# Các giá trị tương ứng với ba dòng trong bảng sales.
sample_line_totals = pa.array(
    [10000, 5000, 30000, 50000, 27000, 2000, 40000, 9000, 80000],
    type=pa.int64(),
)

# Gọi hàm Python trực tiếp và in kết quả.
# Bước này chưa sử dụng DataFusion hoặc câu SQL.
# Kết quả dự kiến: [10000.0, 5000.0, 27000.0].
print(calculate_discounted_totals(sample_line_totals))

[
  10000,
  5000,
  27000,
  45000,
  27000,
  2000,
  36000,
  9000,
  72000
]


In [ ]:
# Tạo đối tượng hàm do người dùng định nghĩa cho DataFusion.
# udf không chạy phép giảm giá ngay ở bước này.
# Nó mô tả hàm cần gọi và cách DataFusion truyền, nhận dữ liệu.
# Đóng gói hàm Python thành UDF để DataFusion có thể sử dụng.
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/user_defined/index.html#datafusion.user_defined.udf
discount_udf = udf(
    # Hàm Python thực hiện phép tính.
    # Truyền tên hàm không có dấu () vì cần cung cấp hàm,
    # chứ không phải gọi hàm ngay để lấy kết quả.
    calculate_discounted_totals,

    # Danh sách kiểu dữ liệu của các đối số đầu vào.
    # Hàm có một đối số nên danh sách có một phần tử.
    # pa.int64() nghĩa là mỗi giá trị đầu vào là số nguyên 64 bit.
    # Khi thực thi, hàm Python nhận dữ liệu theo mảng Arrow.
    [pa.int64()],

    # Kiểu dữ liệu đầu ra của hàm.
    # pa.float64() nghĩa là mỗi giá trị kết quả là số thực 64 bit.
    # Kiểu này phù hợp với mảng được trả về bởi hàm giảm giá.
    pa.float64(),

    # Khai báo rằng cùng một đầu vào luôn cho cùng một kết quả.
    # Điều này đúng vì hàm chỉ tính theo ngưỡng và tỷ lệ cố định,
    # không phụ thuộc thời gian, số ngẫu nhiên hoặc dữ liệu bên ngoài.
    # DataFusion có thể dùng thông tin này khi tối ưu truy vấn.
    "immutable",

    # Tên dùng để gọi hàm trong SQL.
    # Tên này có thể khác tên hàm Python.
    "calculate_discounted_totals",
)

# Đăng ký đối tượng hàm vào môi trường truy vấn ctx.
# Sau bước này, SQL trong ctx mới nhận biết tên hàm đã đăng ký.
# Chỉ tạo discount_udf ở trên chưa đưa hàm vào môi trường truy vấn.
# 
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.register_udf
ctx.register_udf(discount_udf)

In [54]:
# Tạo truy vấn trên bảng sales đã đăng ký với DataFusion.
#
# line_id và product_name:
# Giữ mã dòng và tên sản phẩm để nhận biết từng dòng kết quả.
#
# quantity * unit_price_vnd AS line_total_vnd:
# Tính giá trị ban đầu của dòng hàng và đặt tên cột kết quả.
#
# calculate_discounted_totals(...) AS discounted_total_vnd:
# Gọi hàm riêng đã đăng ký bằng ctx.register_udf(discount_udf).
# DataFusion truyền giá trị dòng hàng vào hàm để tính tiền sau giảm.
# Đây là tên hàm đã khai báo khi tạo udf, không phải tên biến discount_udf.
#
# ORDER BY line_id:
# Sắp xếp theo mã dòng để dễ đối chiếu với dữ liệu ban đầu.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.sql
discount_query = ctx.sql('''
    select
        line_id,
        product_name,
        quantity * unit_price_vnd as line_total_vnd,
        calculate_discounted_totals(
            quantity * unit_price_vnd
        ) as discounted_total_vnd
    from sales
    order by line_id
''')

# Thực thi truy vấn và hiển thị kết quả.
# Việc tính giá trị dòng hàng và gọi hàm giảm giá diễn ra
# trong quá trình DataFusion thực thi truy vấn này.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.show
discount_query.show()

DataFrame()
+---------+--------------+----------------+----------------------+
| line_id | product_name | line_total_vnd | discounted_total_vnd |
+---------+--------------+----------------+----------------------+
| 1       | Bút          | 10000          | 10000.0              |
| 2       | Bút          | 5000           | 5000.0               |
| 3       | Vở           | 30000          | 27000.0              |
+---------+--------------+----------------+----------------------+


In [56]:
# Tạo câu SQL tổng hợp giá trị trước và sau giảm theo từng sản phẩm.
#
# sum(quantity * unit_price_vnd):
# Cộng giá trị ban đầu của các dòng cùng sản phẩm.
#
# sum(calculate_discounted_totals(...)):
# Áp dụng quy tắc giảm giá cho TỪNG DÒNG trước,
# sau đó cộng các giá trị sau giảm trong cùng nhóm sản phẩm.
# Không phải cộng cả nhóm rồi mới xét điều kiện giảm giá.
#
# group by product_name:
# Gom những dòng có cùng tên sản phẩm.
#
# order by product_name:
# Sắp xếp kết quả để hiển thị và kiểm tra theo thứ tự ổn định.
discount_summary_sql = '''
    select
        product_name,
        sum(quantity * unit_price_vnd) as total_revenue_vnd,
        sum(
            calculate_discounted_totals(quantity * unit_price_vnd)
        ) as total_after_discount_vnd
    from sales
    group by product_name
    order by product_name
'''

# Tạo đối tượng truy vấn DataFusion từ chuỗi SQL.
# Tên hàm calculate_discounted_totals phải khớp tên đã đăng ký bằng UDF.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/context/index.html#datafusion.context.SessionContext.sql
discount_summary_query = ctx.sql(discount_summary_sql)

# Thực thi truy vấn và lấy toàn bộ kết quả về dưới dạng các lô Arrow.
# Bước này thực hiện việc gọi hàm giảm giá, gom nhóm và tính tổng.
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.collect
discount_summary_batches = discount_summary_query.collect()

# Tạo một bảng Arrow từ các lô kết quả để dễ hiển thị và kiểm tra.
#
# Tài liệu:
# https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.from_batches
discount_summary_result = pa.Table.from_batches(discount_summary_batches)

# In kết quả đã lấy về.
# print() không chạy lại truy vấn.
print(discount_summary_result)

# Chuyển bảng kết quả thành từ điển Python để đối chiếu giá trị.
#
# Tài liệu:
# https://arrow.apache.org/docs/python/generated/pyarrow.Table.html#pyarrow.Table.to_pydict
actual_discount_summary = discount_summary_result.to_pydict()

# Khai báo kết quả mong đợi được tính bằng tay.
#
# Bút:
# Hai dòng có giá trị 10000 và 5000, đều dưới ngưỡng 30000.
# Tổng trước giảm = 15000; tổng sau giảm = 15000.
#
# Vở:
# Một dòng có giá trị 30000, được giảm 10%.
# Tổng trước giảm = 30000; tổng sau giảm = 27000.
expected_discount_summary = {
    "product_name": ["Bút", "Vở"],
    "total_revenue_vnd": [15000, 30000],
    "total_after_discount_vnd": [15000.0, 27000.0],
}

# Kiểm tra tên cột, thứ tự và giá trị thực tế có đúng như mong đợi không.
# Nếu đúng: tiếp tục chạy.
# Nếu sai: báo AssertionError kèm kết quả để tìm nguyên nhân.
assert actual_discount_summary == expected_discount_summary, (
    f"Kết quả không đúng. "
    f"Mong đợi: {expected_discount_summary}; "
    f"thực tế: {actual_discount_summary}"
)

print("Kiểm tra kết quả thành công.")

pyarrow.Table
product_name: string
total_revenue_vnd: int64
total_after_discount_vnd: double
----
product_name: [["Bút","Vở"]]
total_revenue_vnd: [[15000,30000]]
total_after_discount_vnd: [[15000,27000]]
Kiểm tra kết quả thành công.


In [58]:
# Hiển thị kế hoạch xử lý của truy vấn tổng hợp sau giảm giá.
#
# Kế hoạch cho biết DataFusion sẽ:
# 1. Đọc bảng sales.
# 2. Tính quantity * unit_price_vnd cho từng dòng.
# 3. Gọi hàm calculate_discounted_totals(...) do nhóm đăng ký.
# 4. Gom nhóm theo product_name.
# 5. Tính tổng doanh thu và tổng tiền sau giảm.
# 6. Sắp xếp kết quả theo product_name.
#
# explain() chỉ hiển thị kế hoạch.
# Nó không dùng để lấy bảng doanh thu cuối cùng.
# Muốn lấy kết quả phải dùng discount_summary_query.collect()
# hoặc discount_summary_query.show().
#
# Tài liệu:
# https://datafusion.apache.org/python/autoapi/datafusion/dataframe/index.html#datafusion.dataframe.DataFrame.explain
discount_summary_query.show()
discount_summary_query.explain()

DataFrame()
+--------------+-------------------+--------------------------+
| product_name | total_revenue_vnd | total_after_discount_vnd |
+--------------+-------------------+--------------------------+
| Bút          | 15000             | 15000.0                  |
| Vở           | 30000             | 27000.0                  |
+--------------+-------------------+--------------------------+
DataFrame()
+---------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
| plan_type     | plan                                                                                                                                                                                                                              |
+---------------+-----------------------------------------------------------------------------------